In [1]:
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

CONFIGS_DIR = Path('../configs')
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACTS_ROOT = Path('../artifacts')
FIGURES_EDA = ARTIFACTS_ROOT / 'figures' / 'eda'
FIGURES_EDA.mkdir(parents=True, exist_ok=True)

# 1. Формируем конфигурацию данных на основе текущих настроек ноутбука
data_config = {
    'source_path': '../data/ENB2012_data.csv',
    'separator': ';',
    'decimal': ',',
    'dropna_how': 'all',
    'columns': {
        'numeric': ['X1', 'X2', 'X3', 'X4', 'X5', 'X7'],
        'categorical': ['X6', 'X8'],
        'targets': ['Y1', 'Y2']
    }
}

# 2. Сохраняем в configs/data.yaml
data_cfg_path = CONFIGS_DIR / 'data.yaml'
with open(data_cfg_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_config, f, default_flow_style=False, allow_unicode=True)
print(f'Конфигурация данных сохранена: {data_cfg_path}')

# 3. Загружаем обратно (гарантия консистентности)
with open(data_cfg_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

def save_figure(fig, name):
    filepath = FIGURES_EDA / f'{name}.png'
    fig.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return filepath

Конфигурация данных сохранена: ..\configs\data.yaml


In [2]:
df = pd.read_csv(
    cfg['source_path'],
    sep=cfg['separator'],
    decimal=cfg['decimal']
)
df = df.dropna(how=cfg['dropna_how']).reset_index(drop=True)

num_cols = cfg['columns']['numeric']
cat_cols = cfg['columns']['categorical']
target_cols = cfg['columns']['targets']

df[cat_cols] = df[cat_cols].astype('category')

print(f'Размер данных: {df.shape}')
print(f'Числовые признаки: {num_cols}')
print(f'Категориальные признаки: {cat_cols}')
print(f'Целевые переменные: {target_cols}')

Размер данных: (768, 10)
Числовые признаки: ['X1', 'X2', 'X3', 'X4', 'X5', 'X7']
Категориальные признаки: ['X6', 'X8']
Целевые переменные: ['Y1', 'Y2']


In [3]:
missing = df.isna().sum()
duplicates = df.duplicated().sum()

print(f'Всего пропусков: {missing.sum()}')
print(f'Полные дубликаты строк: {duplicates}')
if missing.sum() == 0 and duplicates == 0:
    print('Данные чистые: пропусков и дубликатов нет')

Всего пропусков: 0
Полные дубликаты строк: 0
Данные чистые: пропусков и дубликатов нет


In [4]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, target in enumerate(target_cols):
    row, col = divmod(idx, 2)
    ax = axes[row, col]
    sns.histplot(df[target], bins=30, kde=True, ax=ax, color='steelblue')
    ax.axvline(df[target].mean(), color='red', linestyle='--', label=f'Среднее: {df[target].mean():.2f}')
    ax.axvline(df[target].median(), color='green', linestyle='--', label=f'Медиана: {df[target].median():.2f}')
    ax.set_title(f'Распределение {target}')
    ax.legend(fontsize=8)

ax = axes[1, 1]
sns.boxplot(data=df[target_cols], ax=ax, palette='Set2')
ax.set_title('Boxplot: сравнение целевых переменных')
plt.tight_layout()
save_figure(fig, 'target_distributions')

WindowsPath('../artifacts/figures/eda/target_distributions.png')

In [5]:
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Корреляционная матрица')
plt.tight_layout()
save_figure(plt.gcf(), 'correlation_matrix')

print('\nТоп-корреляции с целевыми переменными:')
for target in target_cols:
    top = df.corr()[target].drop(target_cols).abs().nlargest(3)
    print(f'{target}: {dict(zip(top.index, top.values.round(3)))}')


Топ-корреляции с целевыми переменными:
Y1: {'X5': np.float64(0.889), 'X4': np.float64(0.862), 'X2': np.float64(0.658)}
Y2: {'X5': np.float64(0.896), 'X4': np.float64(0.863), 'X2': np.float64(0.673)}


In [6]:
target = 'Y2'
top_features = df.corr()[target].drop(target_cols).abs().nlargest(2).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, feat in zip(axes, top_features):
    sns.scatterplot(x=df[feat], y=df[target], alpha=0.6, ax=ax, color='darkblue')
    z = np.polyfit(df[feat], df[target], 1)
    p = np.poly1d(z)
    ax.plot(df[feat], p(df[feat]), 'r--', alpha=0.7)
    ax.set_title(f'{feat} vs {target}')
plt.tight_layout()
save_figure(fig, 'top_features_vs_Y2')

WindowsPath('../artifacts/figures/eda/top_features_vs_Y2.png')

In [7]:
for col in cat_cols:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, t in zip(axes, target_cols):
        sns.boxplot(x=col, y=t, data=df, ax=ax)
        ax.set_title(f'{t} по {col}')
    plt.tight_layout()
    save_figure(fig, f'{col}_boxplots')

In [8]:
plt.figure(figsize=(6, 5))
sns.scatterplot(x='Y1', y='Y2', data=df, alpha=0.6, color='purple')
plt.title('Y2 vs Y1')
plt.xlabel('Y1 (Heating Load)')
plt.ylabel('Y2 (Cooling Load)')
z = np.polyfit(df['Y1'], df['Y2'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Y1'].min(), df['Y1'].max(), 100)
plt.plot(x_line, p(x_line), 'r--', alpha=0.8)
plt.tight_layout()
save_figure(plt.gcf(), 'Y1_vs_Y2')

WindowsPath('../artifacts/figures/eda/Y1_vs_Y2.png')

In [9]:
print('\nИтоговые выводы по EDA:')
print('1. Пропусков и дубликатов нет.')
print(f'2. Y1 и Y2 коррелируют: {df["Y1"].corr(df["Y2"]):+.3f}.')
print(f'3. Топ-предикторы для Y2: {", ".join(df.corr()["Y2"].drop(target_cols).abs().nlargest(3).index.tolist())}.')
print('4. Категориальные признаки показывают различия между группами.')
print(f'5. Все графики сохранены в: {FIGURES_EDA}')


Итоговые выводы по EDA:
1. Пропусков и дубликатов нет.
2. Y1 и Y2 коррелируют: +0.976.
3. Топ-предикторы для Y2: X5, X4, X2.
4. Категориальные признаки показывают различия между группами.
5. Все графики сохранены в: ..\artifacts\figures\eda


In [10]:
# Финальная проверка
print('Проверка воспроизводимости:')
print(f'Размер данных: {df.shape}')
print(f'Пропуски: {df.isna().sum().sum()}')
print(f'Дубликаты: {df.duplicated().sum()}')
for target in target_cols:
    print(f'{target}: диапазон [{df[target].min():.2f}, {df[target].max():.2f}]')
print(f'Создано фигур: {len(list(FIGURES_EDA.glob("*.png")))}')
print('EDA завершено.')

Проверка воспроизводимости:
Размер данных: (768, 10)
Пропуски: 0
Дубликаты: 0
Y1: диапазон [6.01, 43.10]
Y2: диапазон [10.90, 48.03]
Создано фигур: 6
EDA завершено.
